# Session 16 — MLOps Pipeline for Intelligent Surveillance System

**Goal:** build a two-model surveillance pipeline — an **activity classifier** (what
is happening in a frame) plus an **anomaly detector** (does this frame look unlike
anything normal) — and serve both behind a real-time inference endpoint, tracked with
MLflow throughout.

## A note on the dataset

A real surveillance system ingests video frames from cameras. This notebook uses
scikit-learn's built-in `digits` dataset (8x8 grayscale images) as a stand-in for
per-frame image features — small enough to train instantly, but structurally the
same problem (image in, class/anomaly-score out) as a real camera-frame classifier.
Swap in real frames + a CNN feature extractor (see Session 20's medical imaging CNN
for that pattern) for a production system.

## Prerequisites

```bash
pip install mlflow scikit-learn
```
Runs entirely locally.

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, classification_report

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session16-surveillance")

## Step 1 — Activity classification: what is in this frame?

We treat each `digits` class as a stand-in "activity label" (e.g. class 0 = "empty
hallway", class 1 = "person walking", ... — the labels are arbitrary for this demo,
what matters is the pipeline shape).

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
ACTIVITY_NAMES = [f"activity_{i}" for i in range(10)]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train: {len(X_train)}, test: {len(X_test)}, frame shape: {digits.images[0].shape}")

In [ ]:
with mlflow.start_run(run_name="activity_classifier") as run:
    clf = RandomForestClassifier(n_estimators=200, random_state=0)
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.sklearn.log_model(clf, artifact_path="activity_classifier")

    classifier_run_id = run.info.run_id
    print(f"Activity classifier accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=ACTIVITY_NAMES, zero_division=0))

## Step 2 — Anomaly detection: does this frame look unusual?

Separate from classification: an `IsolationForest` learns what "normal" frames look
like and flags outliers, without needing an "anomaly" label at training time (most
surveillance anomalies are rare and unlabeled by definition).

In [ ]:
with mlflow.start_run(run_name="anomaly_detector") as run:
    anomaly_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=0)
    anomaly_model.fit(X_train)

    anomaly_scores = anomaly_model.decision_function(X_test)
    is_anomaly = anomaly_model.predict(X_test) == -1

    mlflow.log_param("contamination", 0.05)
    mlflow.log_metric("pct_flagged_anomalous", is_anomaly.mean())
    mlflow.sklearn.log_model(anomaly_model, artifact_path="anomaly_detector")

    anomaly_run_id = run.info.run_id
    print(f"{is_anomaly.sum()} / {len(is_anomaly)} test frames flagged as anomalous "
          f"({is_anomaly.mean():.1%})")

## Step 3 — Combine both models into one real-time inference function

A production frame-processing loop would call something like this per incoming
frame: classify the activity, and separately check if the frame itself looks
anomalous (e.g. an unrecognized object, unusual lighting, camera tampering).

In [ ]:
import mlflow.sklearn

loaded_classifier = mlflow.sklearn.load_model(f"runs:/{classifier_run_id}/activity_classifier")
loaded_anomaly_model = mlflow.sklearn.load_model(f"runs:/{anomaly_run_id}/anomaly_detector")

def process_frame(frame_features):
    frame_features = frame_features.reshape(1, -1)
    activity_pred = loaded_classifier.predict(frame_features)[0]
    activity_proba = loaded_classifier.predict_proba(frame_features)[0].max()
    anomaly_flag = loaded_anomaly_model.predict(frame_features)[0] == -1
    anomaly_score = loaded_anomaly_model.decision_function(frame_features)[0]

    return {
        "predicted_activity": ACTIVITY_NAMES[activity_pred],
        "activity_confidence": round(float(activity_proba), 4),
        "is_anomalous": bool(anomaly_flag),
        "anomaly_score": round(float(anomaly_score), 4),
    }

for i in range(3):
    result = process_frame(X_test[i])
    print(f"frame {i}: {result}")

## Step 4 — Serve it as a real-time API

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Surveillance Frame Processor")

class FrameRequest(BaseModel):
    pixels: List[float]  # flattened 8x8 = 64 values

@app.post("/v1/process-frame")
def process_frame_endpoint(request: FrameRequest):
    return process_frame(np.array(request.pixels))

client = TestClient(app)
response = client.post("/v1/process-frame", json={"pixels": X_test[0].tolist()})
print(response.status_code, response.json())

## Step 5 — Alerting policy

A frame triggers a human-review alert only when *both* signals agree something is
off — the classifier is unsure AND the anomaly detector flags it — a simple but
effective way to reduce false-alarm fatigue versus alerting on either signal alone.

In [ ]:
def should_alert(result, confidence_threshold=0.5):
    low_confidence = result["activity_confidence"] < confidence_threshold
    return result["is_anomalous"] and low_confidence

flagged = 0
for i in range(len(X_test)):
    result = process_frame(X_test[i])
    if should_alert(result):
        flagged += 1

print(f"{flagged} / {len(X_test)} frames would trigger a human-review alert "
      f"(anomalous AND low classifier confidence).")

## What to try next

* Replace `digits` with real video frames run through a pretrained CNN feature
  extractor (e.g. a ResNet's penultimate layer as the feature vector) — the
  classification/anomaly-detection/alerting code above doesn't need to change.
* Add frame-rate throttling and a rolling buffer so `should_alert` considers a short
  sequence of frames, not a single one, to avoid single-frame false positives (a
  bird crossing the camera for one frame vs. a sustained intrusion).
* Track false-alarm rate over time in the same BigQuery-style monitoring table
  pattern from Session 12, so the `confidence_threshold` can be tuned from real data.